In [1]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt

import sys
import os
from pathlib import Path

# move upward until we find repo root
repo_root = Path.cwd()

while not (repo_root / ".git").exists():
    repo_root = repo_root.parent

sys.path.append(str(repo_root))
from config import config
from DataPipeline.utils.bets_utils import generate_bets 

In [2]:
non_merged = pd.read_csv(r'C:\Users\jcmar\my_files\SportsBetting\Data\non_merged_features\non_merged_stats.csv')
df_ml_history = pd.read_csv(config.ml_history_fp)
df_parlay = pd.read_csv(config.parlay_history_fp)


In [3]:
df_ml_history.shape

(12, 23)

In [3]:
from collections import defaultdict
def american_to_decimal(odds):
    odds = np.asarray(odds)
    return np.where(
        odds > 0, 
        1 + odds / 100, 
        1 + 100 / abs(odds)
    )

def returns_by_date(starting_bankroll=500):
    
    df_ml = pd.read_csv(config.ml_history_fp)
    df_parlay = pd.read_csv(config.parlay_history_fp)

    types = ['open', 'close1', 'close2']

    df_ml = pd.read_csv(config.ml_history_fp)
    df_parlay = pd.read_csv(config.parlay_history_fp)

    bankroll_data = defaultdict(list)
    parlay_data = defaultdict(list)
    ml_data = defaultdict(list)
    
    ml_data['date'] = df_ml['date']
    bankroll_data['date'] = df_parlay.groupby('date')['date'].first()
    parlay_data['date'] = df_parlay.groupby('date')['date'].first()

    for type_ in types: 

        date_index = df_ml['date']

        win_bet = pd.Series(0, index=date_index, dtype='boolean')
        mask_bet = df_ml[f"pred_winner_{type_}"].notna().to_numpy()

        win_bet.iloc[mask_bet] = (
            df_ml.loc[mask_bet, f"pred_winner_{type_}"].astype(int).to_numpy()
            == df_ml.loc[mask_bet, "winner_bool"].astype(int).to_numpy()
        )
        win_bet[~mask_bet] = False

        choice_stake = pd.Series(
            np.array(
                df_ml[f'fstar_{type_}'].fillna(0)
            ), 
        index=date_index
        )
        net_stake = choice_stake.where(win_bet, -choice_stake)
        net_stake = net_stake.where(mask_bet, 0)

        choice_odds = pd.Series(
            np.where(
                df_ml[f'pred_winner_{type_}'] == 1, 
                american_to_decimal(df_ml[f'{type_}_red'])-1, 
                american_to_decimal(df_ml[f'{type_}_blue'])-1
            ),
        index=date_index
        )
        choice_odds = choice_odds.fillna(0)
        net_odds = choice_odds.where(win_bet, -1)
        net_odds = net_odds.where(mask_bet, 0)

        ml_data[f'net_stake_{type_}'] = net_stake.copy()
        ml_data[f'net_odds_{type_}'] = net_odds.copy()


        date_index = df_parlay['date']

        leg_win = pd.Series(
            (df_parlay[f'choice_fighter_bool_{type_}'] == df_parlay[f'winner_bool_{type_}']).to_numpy(), 
            index=date_index
        )
        win_parlay = leg_win.groupby(level=0).all()
        single_date_index = win_parlay.index

        choice_parlay_odds = pd.Series(
            np.where(
                df_parlay[f'choice_fighter_bool_{type_}'] == 1,
                american_to_decimal(df_parlay[f'{type_}_red']),
                american_to_decimal(df_parlay[f'{type_}_blue'])
            ), 
            index=date_index
        ).fillna(0)

        parlay_stake = pd.Series(
            df_parlay.groupby('date')[f'{type_}_fstar'].first().fillna(0),
            index=single_date_index
        )
        parlay_net_stake = parlay_stake.where(win_parlay, -parlay_stake)

        parlay_odds = pd.Series(np.array(choice_parlay_odds.groupby(level=0).prod() - 1), index=single_date_index)
        parlay_net_odds = parlay_odds.where(win_parlay, -parlay_odds)

        parlay_data[f'net_odds_{type_}'] = parlay_net_odds.copy()
        parlay_data[f'net_stake_{type_}'] = parlay_net_stake.copy()
        parlay_data[f'win_parlay_{type_}'] = win_parlay.copy()

        bankroll = starting_bankroll
        bankroll_history = []
        profits = []
        for date in date_index.unique():

            wins = win_bet[mask_bet].loc[date]
            stakes = choice_stake[mask_bet].loc[date]
            odds = choice_odds[mask_bet].loc[date]

            profit_ml = np.where(
                wins, 
                stakes * bankroll * odds, 
                -stakes * bankroll
            )

            wins_parlay = win_parlay.loc[date]
            stakes_parlay = parlay_stake.loc[date]
            odds_parlay = parlay_odds.loc[date]

            profit_parlay = stakes_parlay * odds_parlay * bankroll if wins_parlay else -stakes_parlay * bankroll 

            total_profit = profit_ml.sum() + profit_parlay
            profits.append(total_profit) 

            bankroll += total_profit.sum()
            bankroll_history.append(bankroll)

        bankroll_data[f'bankroll_{type_}'] = bankroll_history
        bankroll_data[f'profits_{type_}'] = profits

    other_info = df_ml[[
        'open_red', 'open_blue', 'close1_red', 'close1_blue', 'close2_red', 'close2_blue',
        'fighter_red', 'fighter_blue', 'pred_winner_open', 'pred_winner_close1', 'pred_winner_close2',
        'winner_bool', 'winner_name'
    ]].reset_index(drop=True)

    ml_results = {
        key: value.to_numpy()
        for key, value in dict(ml_data).items()
    }

    ml_results = pd.DataFrame(ml_results)
    ml_results = pd.concat([ml_results, other_info], axis=1)

    parlay_results = pd.DataFrame(dict(parlay_data))
    bankroll_results = pd.DataFrame(dict(bankroll_data))

    return ml_results, parlay_results, bankroll_results


ml_results, parlay_results, bankroll_results = returns_by_date()


In [4]:
ml_results = pd.read_csv(r'C:\Users\jcmar\my_files\SportsBetting\Data\betting_results\ml_returns.csv')
parlay_results = pd.read_csv(r'C:\Users\jcmar\my_files\SportsBetting\Data\betting_results\parlay_results.csv')

In [12]:
ml_results.columns

Index(['date', 'net_stake_open', 'net_odds_open', 'net_stake_close1',
       'net_odds_close1', 'net_stake_close2', 'net_odds_close2', 'open_red',
       'open_blue', 'close1_red', 'close1_blue', 'close2_red', 'close2_blue',
       'fighter_red', 'fighter_blue', 'pred_winner_open', 'pred_winner_close1',
       'pred_winner_close2', 'winner_bool', 'winner_name'],
      dtype='str')

In [14]:
test_columns = [ 
    'net_stake_open', 'net_stake_close1', 'net_stake_close2', 'date', 'fighter_red', 'fighter_blue',
    'close1_red', 'close1_blue', 'close2_red', 'close2_blue', 
    'pred_winner_open', 'pred_winner_close1', 'pred_winner_close2', 'winner_bool',
]

In [9]:
def analysis(ml_results, type_):
    no_draws = ml_results[ml_results[f'pred_winner_{type_}'] < 2].dropna(subset=[f'{type_}_red', f'{type_}_blue'])
    all_preds = no_draws.dropna(subset=[f'pred_winner_{type_}'])
    bets_only = all_preds[all_preds[f'net_stake_{type_}'] != 0]

    accuracy_all = (all_preds[f'pred_winner_{type_}'] == all_preds['winner_bool']).mean()
    accuracy_bets = (bets_only[f'pred_winner_{type_}'] == bets_only['winner_bool']).mean()

    print(round(accuracy_all, 3), round(accuracy_bets, 3))


    choice_odds = np.where(
        bets_only[f'pred_winner_{type_}'] == 1, 
        bets_only[f'{type_}_red'], 
        bets_only[f'{type_}_blue']
    )

    dog_bets = bets_only[choice_odds > 0]
    fav_bets = bets_only[choice_odds < 0]
    total_dog = dog_bets.shape[0]

    print(choice_odds.shape)

analysis(ml_results, 'open')
analysis(ml_results, 'close1')

0.714 0.738
(126,)
0.683 0.679
(109,)


In [15]:
ml_results[test_columns].tail(20)

,net_stake_open,net_stake_close1,net_stake_close2,date,fighter_red,fighter_blue,close1_red,close1_blue,close2_red,close2_blue,pred_winner_open,pred_winner_close1,pred_winner_close2,winner_bool
255,0.000000,0.000000,0.000000,2026-08-01,dennis buzukja,bogdan grad,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
256,0.145920,0.065244,0.099575,2026-08-01,jan blachowicz,navajo stirling,250.0,-370.0,290.0,-330.0,0.0,0.0,0.0,0.0
257,0.000000,0.000000,0.000000,2026-08-01,oban elliott,michael oliveira,275.0,-910.0,575.0,-350.0,NaN,NaN,NaN,0.0
258,-0.120615,-0.176302,-0.189195,2026-08-01,ludovit klein,tofiq musayev,-275.0,200.0,-250.0,225.0,1.0,1.0,1.0,0.0
259,0.000000,0.000000,0.000000,2026-08-01,milos janicic,noah gugnon,-138.0,-120.0,-105.0,118.0,NaN,NaN,NaN,0.0
260,0.000000,0.000000,0.000000,2026-08-01,vlasto cepo,gilbert urbina,-350.0,230.0,-280.0,275.0,NaN,NaN,NaN,0.0
261,0.152204,0.168451,0.212925,2026-08-01,aleksandar rakic,marcin tybura,-420.0,260.0,-350.0,310.0,1.0,1.0,1.0,1.0
262,-0.085041,-0.000000,-0.000000,2026-08-01,dusko todorovic,robert valentin,125.0,-175.0,140.0,-150.0,1.0,1.0,1.0,0.0
263,0.000000,0.000000,0.000000,2026-08-08,billy ray goff,ty miller,300.0,-500.0,380.0,-400.0,0.0,0.0,0.0,0.0
264,0.080583,0.104996,0.124341,2026-08-08,diego ferreira,billy quarantillo,-188.0,143.0,-175.0,154.0,1.0,1.0,1.0,1.0


In [6]:
test_columns = [
    'date', 'fstar_open', 'fstar_close1', 'fstar_close2',
    'pred_name_close1', 'pred_name_close2', 'pred_name_close1',
    'pred_name_close2',  'winner_bool'
]
df_ml_history[test_columns].tail(20)

,date,fstar_open,fstar_close1,fstar_close2,pred_name_close1,pred_name_close2,pred_name_close1,pred_name_close2,winner_bool
0,2026-08-08,0.051324,0.211296,0.210212,alexia thainara,alexia thainara,alexia thainara,alexia thainara,0.0
1,2026-08-08,0.000000,0.000000,0.000000,quillan salkilld,quillan salkilld,quillan salkilld,quillan salkilld,0.0
2,2026-08-08,0.080583,0.202211,0.201802,diego ferreira,diego ferreira,diego ferreira,diego ferreira,1.0
3,2026-08-08,0.000000,0.000000,0.000000,yadier del valle,yadier del valle,yadier del valle,yadier del valle,0.0
4,2026-08-08,0.062791,0.230644,0.230340,juliana miller,juliana miller,juliana miller,juliana miller,1.0
5,2026-08-08,0.000000,0.000000,0.000000,ty miller,ty miller,ty miller,ty miller,0.0
6,2026-08-08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
7,2026-08-08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
8,2026-08-08,0.000000,0.200152,0.199538,miles johns,miles johns,miles johns,miles johns,1.0
9,2026-08-08,0.073677,0.187935,0.188803,diyar nurgozhay,diyar nurgozhay,diyar nurgozhay,diyar nurgozhay,1.0


In [23]:
df_ml_history.columns

Index(['net_odds_open', 'net_odds_close1', 'net_odds_close2', 'winner_bool',
       'winner_name', 'fighter_red', 'fighter_blue', 'pred_name_open',
       'pred_name_close1', 'pred_name_close2', 'pred_winner_open',
       'pred_winner_close1', 'pred_winner_close2', 'open_red', 'open_blue',
       'close1_red', 'close1_blue', 'close2_red', 'close2_blue', 'fstar_open',
       'fstar_close1', 'fstar_close2', 'date'],
      dtype='str')

In [ ]:
choice_odds = np.where(
    bets_only[f'pred_winner_{type_}'] == 1, 
    bets_only[f'{type_}_red'], 
    bets_only[f'{type_}_blue']
)

dog_bets = bets_only[choice_odds > 0]
fav_bets = bets_only[choice_odds < 0]
total_dog = dog_bets.shape[0]

print(choice_odds.shape)